In [0]:

from pyspark.sql import functions as F

# Definir esquema de trabajo
catalogo = "workspace"
esquema = "ingenieria_datos_juanpa"

tabla_bronze = f"{catalogo}.{esquema}.ventas_bronze"
tabla_silver = f"{catalogo}.{esquema}.ventas_silver"
tabla_resumen = f"{catalogo}.{esquema}.ventas_por_categoria"

# Leer ventas originales
df_bronze = spark.table(tabla_bronze)

print("Registros originales:", df_bronze.count())

display(df_bronze.orderBy("id_venta"))


Registros originales: 18


id_venta,fecha,producto,categoria,cantidad,precio_unitario
1,2026-10-01,Laptop Lenovo,Tecnología,2,2500000
2,2026-10-02,Mouse Logitech,Tecnología,5,85000
2,2026-10-02,Mouse Logitech,Tecnología,5,85000
3,2026-10-03,Teclado mecánico,Tecnología,3,180000
4,2026-10-04,Monitor Samsung,Tecnología,2,900000
5,2026-10-05,Silla de oficina,Hogar,2,450000
6,2026-10-06,Escritorio,Hogar,1,750000
7,2026-10-07,Lámpara LED,Hogar,4,65000
8,2026-10-08,Cuaderno,Papelería,10,12000
9,2026-10-09,Bolígrafo,Papelería,20,3500


In [0]:

# Eliminar registros completamente duplicados
df_sin_duplicados = df_bronze.dropDuplicates()

# Eliminar registros inválidos
df_limpio = df_sin_duplicados.filter(
    F.col("producto").isNotNull() &
    (F.trim(F.col("producto")) != "") &
    (F.col("cantidad") > 0) &
    (F.col("precio_unitario") > 0)
)

# Mostrar cantidades antes y después
print("Registros originales:", df_bronze.count())
print("Sin duplicados:", df_sin_duplicados.count())
print("Registros válidos:", df_limpio.count())

display(df_limpio.orderBy("id_venta"))


Registros originales: 18
Sin duplicados: 17
Registros válidos: 15


id_venta,fecha,producto,categoria,cantidad,precio_unitario
1,2026-10-01,Laptop Lenovo,Tecnología,2,2500000
2,2026-10-02,Mouse Logitech,Tecnología,5,85000
3,2026-10-03,Teclado mecánico,Tecnología,3,180000
4,2026-10-04,Monitor Samsung,Tecnología,2,900000
5,2026-10-05,Silla de oficina,Hogar,2,450000
6,2026-10-06,Escritorio,Hogar,1,750000
7,2026-10-07,Lámpara LED,Hogar,4,65000
8,2026-10-08,Cuaderno,Papelería,10,12000
9,2026-10-09,Bolígrafo,Papelería,20,3500
10,2026-10-10,Carpeta,Papelería,8,8000


In [0]:

# Calcular valor total de cada venta
df_silver = df_limpio.withColumn(
    "total_venta",
    F.col("cantidad") * F.col("precio_unitario")
)

display(df_silver.orderBy("id_venta"))


id_venta,fecha,producto,categoria,cantidad,precio_unitario,total_venta
1,2026-10-01,Laptop Lenovo,Tecnología,2,2500000,5000000
2,2026-10-02,Mouse Logitech,Tecnología,5,85000,425000
3,2026-10-03,Teclado mecánico,Tecnología,3,180000,540000
4,2026-10-04,Monitor Samsung,Tecnología,2,900000,1800000
5,2026-10-05,Silla de oficina,Hogar,2,450000,900000
6,2026-10-06,Escritorio,Hogar,1,750000,750000
7,2026-10-07,Lámpara LED,Hogar,4,65000,260000
8,2026-10-08,Cuaderno,Papelería,10,12000,120000
9,2026-10-09,Bolígrafo,Papelería,20,3500,70000
10,2026-10-10,Carpeta,Papelería,8,8000,64000


In [0]:

# Guardar los datos limpios como tabla Delta
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(tabla_silver)

print("Tabla Silver creada:", tabla_silver)
print("Registros:", spark.table(tabla_silver).count())


Tabla Silver creada: workspace.ingenieria_datos_juanpa.ventas_silver
Registros: 15


In [0]:

# Agrupar por categoría y sumar los valores vendidos
df_resumen = df_silver.groupBy("categoria").agg(
    F.sum("total_venta").alias("total_vendido")
)

# Ordenar de mayor a menor
df_resumen = df_resumen.orderBy(
    F.col("total_vendido").desc()
)

# Guardar resumen como tabla Delta
df_resumen.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(tabla_resumen)

display(df_resumen)


categoria,total_vendido
Tecnología,8205000
Hogar,1910000
Accesorios,1125000
Papelería,254000


In [0]:

# Verificar resultados guardados
silver = spark.table(tabla_silver)
resumen = spark.table(tabla_resumen)

print("Registros Bronze:", spark.table(tabla_bronze).count())
print("Registros Silver:", silver.count())
print("Categorías:", resumen.count())

# Comprobar duplicados idénticos en Silver
duplicados = silver.count() - silver.dropDuplicates().count()
print("Duplicados en Silver:", duplicados)

display(resumen.orderBy(F.col("total_vendido").desc()))


Registros Bronze: 18
Registros Silver: 15
Categorías: 4
Duplicados en Silver: 0


categoria,total_vendido
Tecnología,8205000
Hogar,1910000
Accesorios,1125000
Papelería,254000
